In [1]:
import pandas as pd
import requests
import os

## Create Airports DATAFRAME

## Berlin

In [2]:
import requests

url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.52","lon":"13.4","radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

headers = {
	"x-rapidapi-key": "62dfc8b195msh9e5c1ad51089571p1fd2afjsn7bed998e6f8e",
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'searchBy': {'lat': 52.52, 'lon': 13.4}, 'count': 2, 'items': [{'icao': 'EDDT', 'iata': 'TXL', 'name': 'Berlin -Tegel', 'shortName': '-Tegel', 'municipalityName': 'Berlin', 'location': {'lat': 52.5597, 'lon': 13.287699}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDDB', 'iata': 'BER', 'name': 'Berlin Brandenburg', 'shortName': 'Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}


In [ ]:
import os
import requests
import pandas as pd
import time  # 1. Import the time module

latitudes = [52.5200, 53.5511, 48.1374]
longitudes = [13.4050, 9.9937, 11.5755]

def icao_airport_codes(latitudes, longitudes):
    assert len(latitudes) == len(longitudes)
    
    list_for_df = []

    for i in range(len(latitudes)):
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {
            "lat": latitudes[i],
            "lon": longitudes[i],
            "radiusKm": "50",
            "limit": "5",
            "withFlightInfoOnly": "true"
        }
        headers = {
            "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com",
            "X-RapidAPI-Key": os.getenv("RAPIDAPI_KEY") 
        }

        response = requests.request("GET", url, headers=headers, params=querystring)
        
        if response.status_code == 200:
            response_data = response.json()
            items = response_data.get('items', [])
            if items:
                list_for_df.append(pd.json_normalize(items))
        elif response.status_code == 429:
            print(f"Rate limited on index {i}. Retrying might be needed.")
        else:
            print(f"Error {response.status_code} for index {i}")

        # 2. Pause for 1.5 seconds before making the next API call
        time.sleep(1.5)

    if list_for_df:
        return pd.concat(list_for_df, ignore_index=True)
    else:
        return pd.DataFrame()


In [6]:
# coordinates for Berlin, Hamburg, Munich
latitudes = [52.5200, 53.5511, 48.1374]
longitudes = [13.4050, 9.9937, 11.5755]

icao_airport_codes_df = icao_airport_codes(latitudes, longitudes)
icao_airport_codes_df

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


Prepate DATAFRAME to push to database

In [7]:
airports_to_db = icao_airport_codes_df[["icao", "name", "municipalityName"]]
airports_to_db

,icao,name,municipalityName
0,EDDT,Berlin -Tegel,Berlin
1,EDDB,Berlin Brandenburg,Berlin
2,EDDH,Hamburg,Hamburg
3,EDDM,Munich,Munich


Read the cities table from the database to get the "city_id" column

In [18]:
schema = "wikipedia"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [19]:
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

,city_id,city,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575


Getting the 'City_id' to the airports_to_db_dataframe

In [22]:
cities_airports_merged = cities_df.merge(airports_to_db,
                                   left_on = "city",
                                   right_on = "municipalityName",
                                   how="left")

cities_airports_merged

,city_id,city,country,latitude,longitude,icao,name,municipalityName
0,1,Berlin,Germany,52.5200,13.405,EDDT,Berlin -Tegel,Berlin
1,1,Berlin,Germany,52.5200,13.405,EDDB,Berlin Brandenburg,Berlin
2,2,Hamburg,Germany,53.5500,10.000,EDDH,Hamburg,Hamburg
3,3,Munich,Germany,48.1375,11.575,EDDM,Munich,Munich


In [24]:
# Selecting only the columns we need
airports_to_db = cities_airports_merged[["icao","name", "city_id"]]

In [25]:
airports_to_db

,icao,name,city_id
0,EDDT,Berlin -Tegel,1
1,EDDB,Berlin Brandenburg,1
2,EDDH,Hamburg,2
3,EDDM,Munich,3


In [26]:
airports_to_db.rename(columns={"name": "airport_name"}, inplace=True)

In [27]:
airports_to_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   icao          4 non-null      str  
 1   airport_name  4 non-null      str  
 2   city_id       4 non-null      int64
dtypes: int64(1), str(2)
memory usage: 228.0 bytes


## Preparing and sending the populations table 

Create 'airports' table in the DATABASE
```sql

-- Create the 'airport' table

CREATE TABLE airports(
    icao VARCHAR(10),
    airport_name VARCHAR(255),
    City_id INT NOT NULL,
    PRIMARY KEY (icao),
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);


In [28]:
airports_to_db.to_sql('airports',
                  if_exists='append',
                  con=connection_string,
                  index=False)

4

In [29]:
pd.read_sql("airports", con=connection_string)

,icao,airport_name,city_id
0,EDDB,Berlin Brandenburg,1
1,EDDH,Hamburg,2
2,EDDM,Munich,3
3,EDDT,Berlin -Tegel,1
